# 14 · Tries, Union-Find and LRU Cache

After this notebook you can code the three favourite "design a data structure" answers from memory — a trie with autocomplete, union-find with path compression and union by rank, and an O(1) LRU cache (two ways, plus LFU) — and connect each to AI engineering: tokenizer prefix matching, near-duplicate clustering, and embedding/response caches measured on realistic skewed traffic.

**Time:** ~45 min · **Runs:** offline on CPU in < 1 min · **Needs:** [Hashing and sliding window](03_hashing_and_sliding_window.ipynb), [Linked lists](05_linked_lists.ipynb), [Trees and BST](09_trees_and_bst.ipynb)

## Why this matters in interviews

- "Implement a trie", "number of connected components / redundant connection" and "design an LRU cache" are asked constantly because each is ~20 lines once you know the trick — and impossible to improvise if you do not.
- The LRU cache is *the* O(1) design question: a hash map for lookup plus a doubly linked list for recency. Expect to write the linked-list version even if you mention `OrderedDict`.
- AI-engineering follow-ups land on the same structures: how a subword tokenizer matches the longest vocabulary piece, how a dedup pipeline turns similar pairs into clusters, and what hit rate a cache really gets on skewed traffic (and why hit rate is not cost saved).

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `fo03` | Why do LLMs use a subword tokenizer instead of treating every word as one token? (longest-match subword pieces from a trie; nothing is ever unknown) |
| `dt09` | How do you deduplicate a large document corpus? (exact hashes + union-find clustering of near-duplicate pairs) |
| `po09` | What are the caching layers in a GenAI application? (an embedding cache and a response cache, built on LRU) |
| `pd18` | What is the difference between a cache hit rate and a cost saving? (both measured on the same Zipf traffic) |
| `in42` | How do you keep a cache from serving stale answers when the underlying data changes? (versioned keys and TTL) |

## What interviewers ask

- "Implement a trie with insert / search / startsWith" (LeetCode 208); "design search autocomplete" (642); "replace words" (648); "word search II" (212).
- "Number of provinces / connected components" (547 / 323), "redundant connection" (684), "accounts merge" (721).
- "Design an LRU cache with O(1) get and put" (146); "design an LFU cache" (460).
- Follow-ups: "what does path compression buy you?", "why a *doubly* linked list?", "how would you make it thread-safe / distributed?", "what hit rate would you expect?"

In [ ]:
import hashlib
import heapq
import itertools
import math
import random
from collections import OrderedDict, defaultdict, deque

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "axes.axisbelow": True, "axes.titlesize": 11,
                     "legend.frameon": False})
BLUE, ORANGE, AQUA, YELLOW, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#0b0b0b", "#52514e"
rnd = random.Random(0)

## 1. Trie (prefix tree)

**In plain English:** a tree of characters. Every path from the root spells a prefix, and words that share a prefix share the path. Looking up a word or a prefix costs O(length of the word) — it does not matter whether the trie holds ten words or ten million.

Each node keeps a `children` dict, an `is_end` flag (a word ends here) and, for the extras, how many words pass through it and how often a word was used.

In [ ]:
class TrieNode:
    __slots__ = ("children", "is_end", "passes", "freq")
    def __init__(self):
        self.children, self.is_end, self.passes, self.freq = {}, False, 0, 0

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word, freq=1):
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
            node.passes += 1
        node.is_end = True
        node.freq += freq

    def _walk(self, prefix):
        node = self.root
        for ch in prefix:
            node = node.children.get(ch)
            if node is None:
                return None
        return node

    def search(self, word):
        node = self._walk(word)
        return node is not None and node.is_end

    def starts_with(self, prefix):
        return self._walk(prefix) is not None

    def count_prefix(self, prefix):
        node = self._walk(prefix)
        return 0 if node is None else (node.passes if prefix else sum(c.passes for c in self.root.children.values()))

words = ["car", "card", "care", "cat", "do", "dog"]
trie = Trie()
for w in words:
    trie.insert(w)
assert trie.search("car") and trie.search("dog") and not trie.search("ca") and not trie.search("cards")
assert trie.starts_with("ca") and not trie.starts_with("cow") and trie.starts_with("")
assert trie.count_prefix("car") == 3 and trie.count_prefix("d") == 2 and trie.count_prefix("x") == 0
print("search('car'):", trie.search("car"), "| search('ca'):", trie.search("ca"),
      "| starts_with('ca'):", trie.starts_with("ca"), "| words starting 'car':", trie.count_prefix("car"))

In [ ]:
def draw_trie(root, title):
    pos, leaf_x = {}, [0]
    def layout(node, depth, label, key):
        kids = sorted(node.children.items())
        if not kids:
            x = leaf_x[0]; leaf_x[0] += 1
        else:
            x = np.mean([layout(child, depth + 1, ch, key + ch) for ch, child in kids])
        pos[key] = (x, -depth, label, node.is_end, [key + ch for ch, _ in kids])
        return x
    layout(root, 0, "root", "")
    fig, ax = plt.subplots(figsize=(8, 3.6))
    for key, (x, y, _, _, kids) in pos.items():
        for k in kids:
            ax.plot([x, pos[k][0]], [y, pos[k][1]], color="#b5b3ac", lw=1.4, zorder=1)
    for key, (x, y, label, end, _) in pos.items():
        ax.scatter([x], [y], s=420 if key else 700, zorder=2, linewidths=1.5,
                   color="#fde7dc" if end else "#e7f0ff", edgecolors=ORANGE if end else BLUE)
        ax.text(x, y, label, ha="center", va="center", fontsize=9 if key else 8, zorder=3)
        if end:
            ax.text(x, y - 0.38, key, ha="center", va="top", fontsize=8, color=ORANGE)
    ax.set_ylim(min(p[1] for p in pos.values()) - 0.8, 0.5); ax.axis("off"); ax.set_title(title)
    plt.show()

draw_trie(trie.root, "Trie of car, card, care, cat, do, dog — orange nodes end a word")

| Operation | Trie | `set` of words | sorted list + `bisect` |
|---|---|---|---|
| is `w` a word? | O(L) | O(L) average (hashing) | O(L log N) |
| any word with prefix `p`? | O(len p) | O(N · L) scan | O(L log N) |
| all words with prefix `p` | O(len p + output) | O(N · L) | O(L log N + output) |
| memory | high (a node per character) | low | low |

### Autocomplete: top suggestions for a prefix

Store how often each query was typed at its end node. For a prefix, walk to its node, collect every word below it, and take the top k by frequency (ties alphabetical) with a heap. A production version caches the top k **at every node** so a keystroke costs O(len p) — trading memory for latency. **Brute force:** scan every stored query with `startswith`.

In [ ]:
def autocomplete(trie, prefix, k=3):
    node = trie._walk(prefix)
    if node is None:
        return []
    found, stack = [], [(node, prefix)]
    while stack:
        n, word = stack.pop()
        if n.is_end:
            found.append((n.freq, word))
        stack.extend((child, word + ch) for ch, child in n.children.items())
    return [w for _, w in heapq.nsmallest(k, found, key=lambda fw: (-fw[0], fw[1]))]

query_log = {"how to reset password": 120, "how to return an item": 80, "how to rest": 5,
             "how long is shipping": 60, "refund policy": 150, "refund status": 90, "reset router": 30}
qtrie = Trie()
for q, f in query_log.items():
    qtrie.insert(q, f)

def autocomplete_brute(log, prefix, k=3):
    return [q for q, _ in sorted(((q, f) for q, f in log.items() if q.startswith(prefix)), key=lambda qf: (-qf[1], qf[0]))[:k]]

for prefix in ["how to re", "ref", "re", "how", "zzz", ""]:
    got = autocomplete(qtrie, prefix)
    assert got == autocomplete_brute(query_log, prefix)
    print(f"{prefix!r:12} -> {got}")

### GenAI tie-in: tokenizer vocabularies and longest-prefix matching (`fo03`)

WordPiece-style tokenizers (BERT) split a word by **greedy longest match**: from the current position, take the longest vocabulary piece that matches, then continue. A trie makes "longest match from here" a single walk down the tree. Keep every single character in the vocabulary and **nothing is ever unknown** — a rare word just becomes more pieces. (GPT-style BPE applies learned merge rules instead of longest match, but it shares the property that matters: a fixed subword vocabulary with no out-of-vocabulary words.)

In [ ]:
pieces = ["un", "token", "tokens", "iz", "ization", "ize", "able", "believ", "ably", "re", "run", "ning",
          "the", "model", "s", "ing"] + list("abcdefghijklmnopqrstuvwxyz")
vocab_trie = Trie()
for p in pieces:
    vocab_trie.insert(p)

def longest_match_tokenize(word, trie):
    out, i = [], 0
    while i < len(word):
        node, j, best = trie.root, i, None
        while j < len(word) and word[j] in node.children:     # walk as far as the trie allows
            node = node.children[word[j]]; j += 1
            if node.is_end:
                best = j                                        # remember the longest full piece
        if best is None:
            out.append("[UNK]"); i += 1                         # cannot happen with single chars in vocab
        else:
            out.append(word[i:best]); i = best
    return out

word_vocab = {"the", "model", "tokens", "run"}                  # a word-level vocabulary for contrast
for w in ["tokenization", "unbelievably", "rerunning", "tokenizable", "qwerty"]:
    toks = longest_match_tokenize(w, vocab_trie)
    word_level = w if w in word_vocab else "[UNK]"
    print(f"{w:13} -> subwords {toks}   | word-level: {word_level}")
    assert "".join(toks) == w and "[UNK]" not in toks
assert longest_match_tokenize("tokenization", vocab_trie) == ["token", "ization"]

Rare or made-up words cost more tokens (`qwerty` becomes letters), which is exactly why unusual names, code and non-English text use more tokens per word. See [tokenization: BPE and tiktoken](../07_genai_foundations/01_tokenization_bpe_and_tiktoken.ipynb) for the real tokenizers.

## 2. Union-Find (disjoint set union)

**In plain English:** every item points to a parent; following parents leads to the group's **root**, and two items are in the same group exactly when they have the same root. `union(a, b)` links one root under the other. Two tricks keep the trees flat:

- **Union by rank (or size):** hang the shorter tree under the taller one, so height stays O(log n).
- **Path compression:** after a `find`, point every node on the path straight at the root.

Together, each operation costs amortised O(α(n)) — the inverse Ackermann function, which is at most 4 for any input that fits in the universe. In practice: constant.

```
before find(4):   0 <- 1 <- 2 <- 3 <- 4        after find(4) with compression:   0 <- 1
                                                                                  0 <- 2
                                                                                  0 <- 3
                                                                                  0 <- 4
```

In [ ]:
class DSU:
    def __init__(self, n, by_rank=True, compress=True):
        self.parent, self.rank = list(range(n)), [0] * n
        self.by_rank, self.compress = by_rank, compress
        self.count, self.hops = n, 0                      # groups; parent pointers followed (for the chart)

    def find(self, x):
        root = x
        while self.parent[root] != root:
            root = self.parent[root]; self.hops += 1
        if self.compress:
            while self.parent[x] != root:                 # second pass: point the path at the root
                self.parent[x], x = root, self.parent[x]
        return root

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False                                  # already connected
        if self.by_rank:
            if self.rank[ra] < self.rank[rb]:
                ra, rb = rb, ra
            if self.rank[ra] == self.rank[rb]:
                self.rank[ra] += 1
        self.parent[rb] = ra                              # hang rb's tree under ra
        self.count -= 1
        return True

d = DSU(6)
d.union(0, 1); d.union(2, 3); d.union(1, 3)
assert d.find(0) == d.find(3) and d.find(4) != d.find(0) and d.count == 3
assert d.union(0, 2) is False                             # 0 and 2 were already connected
print("groups after union(0,1), union(2,3), union(1,3):", d.count)

In [ ]:
sizes = [100, 200, 400, 800, 1600]
variants = {"naive (no rank, no compression)": dict(by_rank=False, compress=False),
            "path compression only": dict(by_rank=False, compress=True),
            "rank + path compression": dict(by_rank=True, compress=True)}
hops = {name: [] for name in variants}
for n in sizes:
    ops = [(i + 1, i) for i in range(n - 1)]                     # a chain of unions: the naive worst case
    qrng = random.Random(n)
    queries = [qrng.randrange(n) for _ in range(n)]
    for name, flags in variants.items():
        dsu = DSU(n, **flags)
        for a, b in ops:
            dsu.union(a, b)
        for q in queries:
            dsu.find(q)
        hops[name].append(dsu.hops / (2 * (n - 1) + n))           # average hops per operation

fig, ax = plt.subplots(figsize=(8, 3.8))
for (name, ys), color in zip(hops.items(), [BLUE, ORANGE, AQUA]):
    ax.plot(sizes, ys, color=color, lw=2, marker="o", ms=4, label=name)
ax.set_xscale("log"); ax.set_yscale("log"); ax.legend()
ax.set_xlabel("n (elements)"); ax.set_ylabel("parent hops per operation (log)")
ax.set_title("Union-find after a chain of unions: naive finds walk O(n); compression and rank flatten it")
plt.show()
print({name: round(ys[-1], 2) for name, ys in hops.items()}, "hops per op at n =", sizes[-1])
assert hops["naive (no rank, no compression)"][-1] > 100 * hops["rank + path compression"][-1]

Read the chart: without either trick the unions build one long chain, so a `find` on a random element walks about half of it and cost per operation grows with n. On this workload **either** trick alone fixes it — compression flattens the chain the first time a deep node is found, rank never builds it. Use both: together they guarantee the near-constant bound on *every* sequence of operations, not just this one.

### Problem — Number of provinces (LeetCode 547) and redundant connection (684)

**Provinces:** `is_connected[i][j] = 1` if cities i and j are directly connected; count the groups. Union every connected pair; the answer is `dsu.count`. (BFS/DFS from [notebook 11](11_graphs_bfs_dfs_topo_dijkstra.ipynb) works too and is the cross-check.)

**Redundant connection:** edges of a tree plus one extra edge arrive in order; return the first edge whose endpoints are **already** connected — it closes a cycle. `union` returning `False` is exactly that test.

In [ ]:
def find_circle_num(is_connected):
    n = len(is_connected)
    dsu = DSU(n)
    for i in range(n):
        for j in range(i + 1, n):
            if is_connected[i][j]:
                dsu.union(i, j)
    return dsu.count

def provinces_bfs(is_connected):
    n, seen, groups = len(is_connected), set(), 0
    for s in range(n):
        if s in seen:
            continue
        groups += 1; seen.add(s); queue = deque([s])
        while queue:
            u = queue.popleft()
            for v in range(n):
                if is_connected[u][v] and v not in seen:
                    seen.add(v); queue.append(v)
    return groups

def redundant_connection(edges):
    dsu = DSU(len(edges) + 1)
    for u, v in edges:
        if not dsu.union(u, v):
            return [u, v]

assert find_circle_num([[1, 1, 0], [1, 1, 0], [0, 0, 1]]) == 2 and find_circle_num([[1, 0, 0], [0, 1, 0], [0, 0, 1]]) == 3
for _ in range(200):
    n = rnd.randint(1, 12)
    m = [[1 if i == j else 0 for j in range(n)] for i in range(n)]
    for _ in range(rnd.randint(0, n)):
        i, j = rnd.randrange(n), rnd.randrange(n)
        m[i][j] = m[j][i] = 1
    assert find_circle_num(m) == provinces_bfs(m)
assert redundant_connection([[1, 2], [1, 3], [2, 3]]) == [2, 3]
assert redundant_connection([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]) == [1, 4]
print("provinces: union-find == BFS on 200 random maps | redundant edge:", redundant_connection([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]))

### GenAI tie-in: clustering near-duplicate documents (`dt09`)

A dedup pipeline has two stages: **exact** duplicates by hashing the normalised text, then **near** duplicates by comparing shingle sets (Jaccard similarity). At scale MinHash + LSH proposes the candidate pairs; here the corpus is tiny, so we compare all pairs. Either way the pairs above a threshold must become **clusters** — "A ≈ B and B ≈ C" means one group — and that is union-find. Keep one document per cluster.

In [ ]:
docs = ["How do I reset my password on the mobile app",
        "How do I reset my password on the mobile app?",          # exact duplicate after normalising
        "How can I reset my password on the mobile app",           # near duplicate
        "Refund policy: refunds take 5 business days to process",
        "Refund policy - refunds take 5 business days to be processed",
        "Our office is closed on public holidays",
        "Shipping is free for orders over 50 dollars"]

def normalise(text):
    return " ".join("".join(ch for ch in text.lower() if ch.isalnum() or ch.isspace()).split())

def shingles(text, k=2):
    words = normalise(text).split()
    return {tuple(words[i:i + k]) for i in range(len(words) - k + 1)}

def jaccard(a, b):
    return len(a & b) / len(a | b)

exact_hash = [hashlib.sha256(normalise(t).encode()).hexdigest()[:10] for t in docs]
print("exact duplicates by hash:", [(i, j) for i, j in itertools.combinations(range(len(docs)), 2) if exact_hash[i] == exact_hash[j]])

sets = [shingles(t) for t in docs]
dsu = DSU(len(docs))
for i, j in itertools.combinations(range(len(docs)), 2):
    sim = jaccard(sets[i], sets[j])
    if sim >= 0.5:
        dsu.union(i, j)
        print(f"  near-duplicate pair ({i}, {j}): Jaccard {sim:.2f}")
clusters = defaultdict(list)
for i in range(len(docs)):
    clusters[dsu.find(i)].append(i)
groups = sorted(clusters.values())
print("clusters:", groups, "-> keep", [g[0] for g in groups])
assert groups == [[0, 1, 2], [3, 4], [5], [6]]

One caution worth saying in the interview: union-find is **transitive**, so a chain of pairwise-similar documents can merge two documents that are not similar to each other at all. Watch cluster sizes, and re-check big clusters against their representative.

## 3. LRU cache (LeetCode 146)

**In plain English:** keep at most `capacity` items; when full, throw out the one that was **used least recently**. Both `get` and `put` must be O(1).

**The trick:** a hash map finds a key's node in O(1); a **doubly linked list** keeps the nodes in recency order and can unlink any node in O(1) because each node knows its neighbours. Most recent at the front, least recent at the back. Two sentinel nodes remove all the empty-list special cases.

```
 map: key -> node
            head <-> [k3] <-> [k1] <-> [k7] <-> tail
                   most recent          least recent (evicted next)
 get(k7): unlink k7, re-insert after head      put when full: evict the node before tail
```

### Way 1 — `OrderedDict` (what you would ship in Python)

In [ ]:
class LRUCacheOD:
    def __init__(self, capacity):
        self.capacity, self.data = capacity, OrderedDict()

    def get(self, key):
        if key not in self.data:
            return -1
        self.data.move_to_end(key)                 # now the most recent
        return self.data[key]

    def put(self, key, value):
        if key in self.data:
            self.data.move_to_end(key)
        self.data[key] = value
        if len(self.data) > self.capacity:
            self.data.popitem(last=False)          # evict the least recent

def run_lc146(cache_cls):
    c = cache_cls(2)
    out = [None, None]
    c.put(1, 1); c.put(2, 2)
    out += [c.get(1)]
    c.put(3, 3)                                    # evicts key 2
    out += [c.get(2)]
    c.put(4, 4)                                    # evicts key 1
    out += [c.get(1), c.get(3), c.get(4)]
    return out

print("LeetCode 146 example:", run_lc146(LRUCacheOD)[2:])
assert run_lc146(LRUCacheOD)[2:] == [1, -1, -1, 3, 4]

### Way 2 — hash map + doubly linked list (the interview implementation)

In [ ]:
class Node:
    __slots__ = ("key", "value", "prev", "next")
    def __init__(self, key=None, value=None):
        self.key, self.value, self.prev, self.next = key, value, None, None

class LRUCache:
    def __init__(self, capacity):
        self.capacity, self.map = capacity, {}
        self.head, self.tail = Node(), Node()               # sentinels
        self.head.next, self.tail.prev = self.tail, self.head

    def _unlink(self, node):
        node.prev.next, node.next.prev = node.next, node.prev

    def _push_front(self, node):
        node.prev, node.next = self.head, self.head.next
        self.head.next.prev = node
        self.head.next = node

    def get(self, key):
        node = self.map.get(key)
        if node is None:
            return -1
        self._unlink(node); self._push_front(node)            # mark as most recent
        return node.value

    def put(self, key, value):
        node = self.map.get(key)
        if node is not None:
            node.value = value
            self._unlink(node); self._push_front(node)
            return
        if len(self.map) == self.capacity:
            lru = self.tail.prev                               # least recent
            self._unlink(lru); del self.map[lru.key]
        node = Node(key, value)
        self.map[key] = node; self._push_front(node)

    def keys_by_recency(self):
        out, node = [], self.head.next
        while node is not self.tail:
            out.append(node.key); node = node.next
        return out

assert run_lc146(LRUCache)[2:] == [1, -1, -1, 3, 4]
a, b = LRUCache(3), LRUCacheOD(3)                            # differential test: 20,000 random operations
for step in range(20_000):
    key = rnd.randint(0, 6)
    if rnd.random() < 0.5:
        assert a.get(key) == b.get(key)
    else:
        a.put(key, step); b.put(key, step)
    assert a.keys_by_recency() == list(reversed(b.data))
print("linked-list LRU == OrderedDict LRU on 20,000 random get/put operations")

**Complexity:** O(1) per `get` and `put`, O(capacity) memory. Why *doubly* linked: removing a node from the middle needs its predecessor, and a singly linked list would have to search for it. Why store the key in the node: on eviction you must delete the key from the map. For memoising a pure function, `functools.lru_cache(maxsize=...)` is this same idea as a decorator ([notebook 08](08_recursion_and_backtracking.ipynb)).

### LFU, briefly (LeetCode 460)

**Least frequently used** evicts the key with the fewest accesses (ties: least recent). The O(1) design: `key → (value, freq)`, `freq → OrderedDict of keys` (recency order within a frequency), and `min_freq`. LFU keeps long-term favourites that LRU would drop after a burst of one-off requests — but it adapts slowly when popularity changes.

In [ ]:
class LFUCache:
    def __init__(self, capacity):
        self.capacity, self.items = capacity, {}                 # key -> [value, freq]
        self.by_freq, self.min_freq = defaultdict(OrderedDict), 0

    def _touch(self, key):
        value, f = self.items[key]
        del self.by_freq[f][key]
        if not self.by_freq[f] and self.min_freq == f:
            self.min_freq = f + 1
        self.items[key][1] = f + 1
        self.by_freq[f + 1][key] = None

    def get(self, key):
        if key not in self.items:
            return -1
        self._touch(key)
        return self.items[key][0]

    def put(self, key, value):
        if self.capacity == 0:
            return
        if key in self.items:
            self.items[key][0] = value; self._touch(key)
            return
        if len(self.items) == self.capacity:
            evict, _ = self.by_freq[self.min_freq].popitem(last=False)   # least frequent, then least recent
            del self.items[evict]
        self.items[key] = [value, 1]
        self.by_freq[1][key] = None
        self.min_freq = 1

c = LFUCache(2)
c.put(1, 1); c.put(2, 2)
trace = [c.get(1)]
c.put(3, 3)                                  # evicts 2 (freq 1)
trace += [c.get(2), c.get(3)]
c.put(4, 4)                                  # 1 and 3 both have freq 2; 1 is least recent -> evicted
trace += [c.get(1), c.get(3), c.get(4)]
print("LeetCode 460 example:", trace)
assert trace == [1, -1, 3, -1, 3, 4]

## 4. GenAI tie-in: LRU for embedding and response caches

Real query traffic is **skewed**: a few questions are asked constantly, most are rare. A Zipf distribution (the i-th most popular item has probability ∝ 1/i) is the standard model. Let us replay 30,000 requests over 5,000 distinct queries through caches of different sizes, and compare with uniform traffic and with an oracle that pins the most popular queries (it needs to know the future popularity, so treat it as a reference line).

In [ ]:
zrng = np.random.default_rng(0)
n_queries, n_requests = 5_000, 30_000
popularity = 1 / np.arange(1, n_queries + 1)
popularity /= popularity.sum()
zipf_trace = zrng.choice(n_queries, size=n_requests, p=popularity).tolist()
uniform_trace = zrng.integers(0, n_queries, n_requests).tolist()

def hit_rate(cache, trace):
    hits = 0
    for q in trace:
        if cache.get(q) != -1:
            hits += 1
        else:
            cache.put(q, True)
    return hits / len(trace)

capacities = [10, 25, 50, 100, 250, 500, 1000, 2500]
results = {"LRU, Zipf traffic": [hit_rate(LRUCacheOD(c), zipf_trace) for c in capacities],
           "LFU, Zipf traffic": [hit_rate(LFUCache(c), zipf_trace) for c in capacities],
           "LRU, uniform traffic": [hit_rate(LRUCacheOD(c), uniform_trace) for c in capacities]}
oracle = [popularity[:c].sum() for c in capacities]
for name, rates in results.items():
    print(f"{name:22}", " ".join(f"{r:.2f}" for r in rates))
print(f"{'pin top-C (oracle)':22}", " ".join(f"{r:.2f}" for r in oracle))
i100 = capacities.index(100)
assert results["LRU, Zipf traffic"][i100] > 5 * results["LRU, uniform traffic"][i100]

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4))
for (name, rates), color in zip(results.items(), [BLUE, ORANGE, AQUA]):
    ax.plot(capacities, rates, color=color, lw=2, marker="o", ms=4, label=name)
ax.plot(capacities, oracle, color=MUTED, lw=1.5, label="pin the top-C queries (oracle reference)")
ax.set_xscale("log"); ax.set_ylim(0, 1); ax.legend(loc="upper left")
ax.set_xlabel("cache capacity (entries, log scale)"); ax.set_ylabel("hit rate")
ax.set_title("Hit rate vs capacity: skewed traffic makes a small cache worthwhile")
plt.show()

Read the chart: with Zipf traffic a cache holding 2% of the distinct queries (100 of 5,000) already catches a large share of requests, and each doubling adds a roughly constant amount — diminishing returns. Uniform traffic gets only `capacity / 5,000`. LFU edges out LRU on a *stationary* Zipf stream because it remembers long-run favourites; LRU recovers faster when popularity shifts, which is why it is the common default.

### Hit rate is not cost saved (`pd18`)

Assume popular questions are short FAQ lookups and rare ones are long RAG or agent requests (the realistic shape). Weight each hit by the cost of the request it avoided.

In [ ]:
rank = np.arange(n_queries)
cost_per_query = 1 + 20 * rank / n_queries          # tokens-ish: rank 0 is cheap, the long tail is expensive
cache, hits, saved, total = LRUCacheOD(250), 0, 0.0, 0.0
for q in zipf_trace:
    total += cost_per_query[q]
    if cache.get(q) != -1:
        hits += 1; saved += cost_per_query[q]
    else:
        cache.put(q, True)
print(f"capacity 250: hit rate {hits / len(zipf_trace):.0%} but cost saved only {saved / total:.0%}")
assert saved / total < hits / len(zipf_trace)

### Keys that include versions, plus a TTL (`in42`, `po09`)

Two cache layers from a real GenAI app, built on the LRU above:

- **Embedding cache** — key `(embedding model, sha256(text))`: identical chunks are never embedded twice, and switching models can never return a vector from the old one.
- **Response cache** — key `(normalised question, prompt version, document version)`: when a source document changes, its version changes, and old answers simply stop matching — no purge logic. A **TTL** bounds staleness for anything the key does not capture.

In [ ]:
class TTLLRUCache(LRUCacheOD):
    """LRU whose entries also expire `ttl` seconds after they were written (clock is injectable for tests)."""
    def __init__(self, capacity, ttl, clock):
        super().__init__(capacity)
        self.ttl, self.clock = ttl, clock

    def get(self, key):
        entry = super().get(key)
        if entry == -1:
            return -1
        value, written = entry
        if self.clock() - written > self.ttl:
            del self.data[key]                                  # expired: treat as a miss
            return -1
        return value

    def put(self, key, value):
        super().put(key, (value, self.clock()))

now = [0.0]
clock = lambda: now[0]
embed_calls = 0
def fake_embed(text):                                           # stands in for a paid embedding call
    global embed_calls
    embed_calls += 1
    return [len(text), text.count(" ")]

emb_cache = TTLLRUCache(1000, ttl=86_400, clock=clock)
def cached_embed(text, model="embed-v1"):
    key = (model, hashlib.sha256(text.encode()).hexdigest())
    vec = emb_cache.get(key)
    if vec == -1:
        vec = fake_embed(text); emb_cache.put(key, vec)
    return vec

chunks = ["refunds take 5 days", "shipping is free", "refunds take 5 days", "shipping is free", "office hours"]
for ch in chunks:
    cached_embed(ch)
cached_embed("office hours", model="embed-v2")                  # new model -> new key -> recomputed
print(f"embedding cache: {len(chunks) + 1} requests -> {embed_calls} real embedding calls")
assert embed_calls == 4

resp = TTLLRUCache(1000, ttl=3600, clock=clock)
doc_version = {"refund_policy": 7}
def response_key(question):
    return (normalise(question), "prompt-v3", doc_version["refund_policy"])
resp.put(response_key("How long do refunds take?"), "Refunds take 5 days.")
print("same question, new wording -> hit:", resp.get(response_key("how long do refunds take")))
doc_version["refund_policy"] = 8                                # the policy document was edited
print("after the document changes -> ", resp.get(response_key("How long do refunds take?")))
resp.put(response_key("How long do refunds take?"), "Refunds take 3 days.")
now[0] += 3601                                                  # an hour later
print("after the TTL expires ->", resp.get(response_key("How long do refunds take?")))
assert resp.get(response_key("How long do refunds take?")) == -1

Normalisation turned two wordings into one exact key; it cannot catch paraphrases — that is the semantic cache layer in [caching: exact and semantic](../16_production/03_caching_exact_and_semantic.ipynb).

## Try it yourself

**1. Replace words (LeetCode 648).** Given roots `["cat", "bat", "rat"]`, replace every word in a sentence by its **shortest** root prefix: `"the cattle was rattled by the battery"` → `"the cat was rat by the bat"`. Hint: walk the trie and stop at the first `is_end`.

In [ ]:
# Solution — try it yourself first, then run this
def replace_words(roots, sentence):
    t = Trie()
    for r in roots:
        t.insert(r)
    out = []
    for word in sentence.split():
        node, prefix = t.root, None
        for i, ch in enumerate(word):
            node = node.children.get(ch)
            if node is None:
                break
            if node.is_end:
                prefix = word[:i + 1]; break          # shortest root wins
        out.append(prefix or word)
    return " ".join(out)

assert replace_words(["cat", "bat", "rat"], "the cattle was rattled by the battery") == "the cat was rat by the bat"
assert replace_words(["a", "b", "c"], "aadsfasf absbs bbab cadsfafs") == "a a b c"
print(replace_words(["cat", "bat", "rat"], "the cattle was rattled by the battery"))

**2. Longest common prefix (LeetCode 14) with a trie.** Insert every string, then walk down from the root while the node has exactly one child and is not the end of a word.

In [ ]:
# Solution — try it yourself first, then run this
def longest_common_prefix(strs):
    if not strs:
        return ""
    t = Trie()
    for s in strs:
        t.insert(s)
    node, prefix = t.root, []
    while len(node.children) == 1 and not node.is_end:
        (ch, node), = node.children.items()
        prefix.append(ch)
    return "".join(prefix)

assert longest_common_prefix(["flower", "flow", "flight"]) == "fl"
assert longest_common_prefix(["dog", "racecar", "car"]) == "" and longest_common_prefix(["same", "same"]) == "same"
assert longest_common_prefix(["ab", "a"]) == "a" and longest_common_prefix([]) == ""
print(longest_common_prefix(["interview", "internet", "interval"]))

**3. Number of islands with union-find.** Redo LeetCode 200 ([notebook 11](11_graphs_bfs_dfs_topo_dijkstra.ipynb)) with a DSU: start with one group per land cell and union each land cell with its land neighbours to the right and below.

In [ ]:
# Solution — try it yourself first, then run this
def num_islands_dsu(grid):
    rows, cols = len(grid), len(grid[0])
    dsu = DSU(rows * cols)
    water = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 0:
                water += 1
                continue
            for nr, nc in ((r + 1, c), (r, c + 1)):
                if nr < rows and nc < cols and grid[nr][nc] == 1:
                    dsu.union(r * cols + c, nr * cols + nc)
    return dsu.count - water                              # every water cell is its own singleton group

def num_islands_bfs(grid):
    rows, cols, seen, count = len(grid), len(grid[0]), set(), 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 1 and (r, c) not in seen:
                count += 1; seen.add((r, c)); queue = deque([(r, c)])
                while queue:
                    i, j = queue.popleft()
                    for ni, nj in ((i + 1, j), (i - 1, j), (i, j + 1), (i, j - 1)):
                        if 0 <= ni < rows and 0 <= nj < cols and grid[ni][nj] == 1 and (ni, nj) not in seen:
                            seen.add((ni, nj)); queue.append((ni, nj))
    return count

assert num_islands_dsu([[1, 1, 0, 0, 0], [1, 1, 0, 0, 0], [0, 0, 1, 0, 0], [0, 0, 0, 1, 1]]) == 3
for _ in range(100):
    g = [[1 if rnd.random() < 0.5 else 0 for _ in range(8)] for _ in range(6)]
    assert num_islands_dsu(g) == num_islands_bfs(g)
print("union-find islands == BFS islands on 100 random grids")

## Say it in an interview

**Trie:** "A tree keyed by characters; insert, search and prefix queries cost O(length of the word), independent of how many words are stored. For autocomplete I store frequencies at word ends and, if latency matters, cache the top k at each node. Memory is the cost — a node per character — so for a static vocabulary a sorted array with binary search or a compressed trie is often enough."

**Union-find:** "Each element points to a parent; the root names the group. With union by rank and path compression every operation is amortised inverse-Ackermann — effectively constant. I reach for it for dynamic connectivity: components, cycle detection while adding edges (redundant connection, Kruskal's MST), and clustering pairs into groups in dedup pipelines."

**LRU:** "A hash map from key to node plus a doubly linked list in recency order, with sentinels. `get` moves the node to the front; `put` inserts at the front and evicts from the back when full. Both O(1). In Python I would ship `OrderedDict` with `move_to_end` and `popitem(last=False)`."

**Caching numbers from this notebook's run:** on Zipf traffic a cache of 2% of the distinct queries catches a large share of requests while uniform traffic gets almost nothing, and the hit rate overstates the cost saved when hits are the cheap requests.

**Follow-ups and traps:**
- *Thread safety:* a lock around get/put (or sharded caches); for many processes, Redis with `maxmemory-policy allkeys-lru` (approximate LRU by sampling).
- *Stale data:* put versions into the key and add a TTL; never cache a failed or low-confidence answer.
- *Union-find:* forgetting path compression and union by rank gives O(n) chains; `union` should report "already connected".
- *Trie:* `search` must check `is_end` (`"ca"` is a prefix, not a word).
- *LFU vs LRU:* LFU wins on stable popularity, LRU adapts faster to change.

## Pattern cheat sheet

| Signal | Structure | Key operations | Cost |
|---|---|---|---|
| prefix queries, autocomplete, dictionary of words | trie | walk children, check `is_end` | O(L) |
| longest vocabulary match from a position | trie walk, remember last `is_end` | — | O(longest piece) |
| "are these connected?", groups that only merge | union-find | `find` with compression, `union` by rank | ~O(1) amortised |
| first edge that closes a cycle | union-find | `union` returns False | ~O(1) per edge |
| cluster similar pairs | union-find over candidate pairs | union each pair, group by root | ~O(pairs) |
| bounded cache, evict least recent | hash map + doubly linked list / `OrderedDict` | move to front, evict tail | O(1) |
| evict least frequent | key → (value, freq), freq → ordered keys, `min_freq` | touch = move up a frequency | O(1) |
| cache that must not go stale | versioned key + TTL | — | O(1) |

## Next

- **[15 · Top 100 LeetCode questions, part 1](15_top100_part1_arrays_hashing_two_pointers.ipynb)** — the patterns are done; now solve the 100 most-asked easy and medium questions, each with an animated answer.
- [Caching: exact and semantic](../16_production/03_caching_exact_and_semantic.ipynb) — the production cache layers, including the semantic one.
- [Tokenization: BPE and tiktoken](../07_genai_foundations/01_tokenization_bpe_and_tiktoken.ipynb) — real subword tokenizers.
- [11 · Graphs](11_graphs_bfs_dfs_topo_dijkstra.ipynb) · [08 · Recursion and backtracking](08_recursion_and_backtracking.ipynb) (`lru_cache`) · [Linked lists](05_linked_lists.ipynb)
- Theory: [DSA basics course](../../dsa_basics/index.html) · [AI system design concepts](../../ai_system_design_concepts/index.html) · [interview bank](../../ai_interview_prep/index.html)